# MeguvizCAC v1.0.0-ApexLoop 全量训练（Bundle v6 <= 2.0 MB · `TunnelSymplect × DynaCore-MoLE × 零重算三段级联`）

完整落地 `docs/roadmap-v1.0.0-apex-symplectloop.md` 与 `v0.9.0-R1` 严格盲测 (`OjinjinBench v1 & v2`) + 双向对比复盘补丁（commit `5bcc3c4`+）：
1. **嵌套 Loop 穿透 (`P2 + P3 + P4`)**：`P4 Tunnel-TRM`（`||J||_2 = 1.0` 活梯度等距回传）+ `P2` 双子空间穿透门 + `P3 L0-Haar` 高频接缝微 KV。
2. **次级 Loop 联结 (`SymplectLoop C1–C4`)**：`C1` 语义/片段 6↔6 有理 Cayley `SO(2)` 精确正交旋转 + `C2` 跨周期动量与惊奇门 + `C3` 零 autograd 内环解析梯度 EBT（直接以 `4B` 门控原生融合进主 Logit，正式退役外部 64 KB EBT）+ `C4` 零冷启动多域上下文环。
3. **Loop 专家组 (`E3 DynaCore-MoLE`)**：`1 Shared + 8 Routed` 纯 FP16 `24×24` 核心 + `288 B` 动力学控制谱 + DeepSeek-V3 无辅助损失死区均衡（零 Gather、零双线性污染、Stage-3a 纯 FP16 零漂移精修）。
4. **零重算三段级联与盲测四连补丁 (`H1–H4 + Top-FP Mining`)**：
   - 选择性 ONNX INT8（排除 `Gather` 与递归环敏感投影，消除 `-0.023` ONNX ROC 掉点）；
   - 外部 Gemini Flash 保留 `20%` `v0.6-R3` 软标签护栏 + **Logit 空间双确认 Top-FP 天花板损失 + Stage 1.5 在线委员会硬负挖掘**（根治自然分布 Top-5 假阳真人冲顶）；
   - OOD 安全 Rung-1 (`<= 0.12`) & Rung-2 (`LTT + 内环 EBT 平衡态门控`) 双重认证 + 灰区难样本自动重入 `Session R3 (T=3)` 深度反思。

> **运行顺序**：选择 **G4 / A100 GPU** 运行时 → 依次运行 **Cell 1（环境与语料恢复）→ Cell 2（核对/解包 80k+12.7k 数据包）→ Cell 3（启动后台抗断线全量训练并自动双写 Drive）**；训练开始后切勿切换运行时，断线重连后运行 **Cell 4** 即可查看进度或补下载。


In [ ]:
# Cell 1: 挂载 Google Drive、核查 GPU、拉取 v1.0.0 最新代码、自动恢复 /content/data/ 冻结资产
import hashlib, json, os, shutil, subprocess, time, torch
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
DRIVE_OUT = Path('/content/drive/MyDrive/MeguvizCAC/v100_apexloop')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
(DRIVE_OUT / '_session_check.txt').write_text(
    f'mounted_utc={time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}\n', encoding='utf-8'
)

if not torch.cuda.is_available():
    raise RuntimeError('未检测到 GPU！请先在右上角选择 G4 / A100 GPU 运行时并连接后再运行本单元。')
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!free -h | head -n 2
print('GPU:', torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory/(1024**3):.1f} GB | bf16={torch.cuda.is_bf16_supported()}')

REPO_DIR = '/content/MeguvizCAC'
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/jichuo1/MeguvizCAC.git {REPO_DIR}
else:
    !git -C {REPO_DIR} -c core.fileMode=false fetch origin && git -C {REPO_DIR} -c core.fileMode=false pull --ff-only
%cd /content/MeguvizCAC
!git log -3 --oneline
!pip -q install -U tokenizers onnx onnx_ir ml_dtypes onnxruntime transformers scipy scikit-learn huggingface_hub

DATA_DIR = Path('/content/data')
for d in (DATA_DIR / 'stage2', DATA_DIR / 'co-teacher-gate'):
    d.mkdir(parents=True, exist_ok=True)
REL = 'https://github.com/jichuo1/MeguvizCAC/releases/download'

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for c in iter(lambda: f.read(1 << 20), b''):
            h.update(c)
    return h.hexdigest()

def ensure_dl(url: str, dst: Path, pref: str | None = None):
    if dst.is_file() and dst.stat().st_size > 0 and (pref is None or sha256_file(dst).startswith(pref)):
        print(f'[cache hit] {dst.name}')
        return
    print(f'[download] {dst.name} ...')
    subprocess.run(['curl', '-fsSL', url, '-o', str(dst)], check=True)
    if pref:
        assert sha256_file(dst).startswith(pref), f'SHA mismatch: {dst.name}'

ensure_dl(f'{REL}/v0.2-data/stage2_distill_bundle.zip', DATA_DIR / 'stage2_distill_bundle.zip')
!unzip -qo /content/data/stage2_distill_bundle.zip -d /content/data/stage2
ensure_dl(f'{REL}/v0.2-qwen-raw/stage2_qwen_bundle.zip', DATA_DIR / 'stage2_qwen_bundle.zip')
!unzip -qoj /content/data/stage2_qwen_bundle.zip -d /content/data/stage2
for name, dst, pref in [
    ('v04-qwen38-twins.jsonl', DATA_DIR / 'v04-qwen38-twins.jsonl', '7ff4227c'),
    ('v04-inkling-wk-pairs.jsonl', DATA_DIR / 'v04-inkling-wk-pairs.jsonl', '0e877a7c'),
    ('v04_semantic_intent_cache.npz', DATA_DIR / 'v04_semantic_intent_cache.npz', '9880e38b'),
    ('v04-merged-1m4.jsonl.gz', DATA_DIR / 'v04-merged-1m4.jsonl.gz', '11b63db0'),
    ('v04-twins-clean-pairs.jsonl', DATA_DIR / 'co-teacher-gate' / 'v04-twins-clean-pairs.jsonl', '3cae814d'),
]:
    ensure_dl(f'{REL}/v0.4-data/{name}', dst, pref)

r3_corpus = DATA_DIR / 'v04-merged-1m4-r3.jsonl.gz'
r3_rep = DATA_DIR / 'r3_persona_labelfix_report.json'
if not (r3_corpus.is_file() and r3_rep.is_file()):
    subprocess.run([
        'python3', '-m', 'meguviz_cac.v04_persona_labelfix',
        '--in', str(DATA_DIR / 'v04-merged-1m4.jsonl.gz'),
        '--out', str(r3_corpus),
        '--report', str(r3_rep),
    ], check=True)

# 预取 BGE-small-zh-v1.5 供 Matryoshka Top-128 SVD 初始化与教师前向使用
!python3 -c "from transformers import AutoModel, AutoTokenizer; AutoTokenizer.from_pretrained('BAAI/bge-small-zh-v1.5'); AutoModel.from_pretrained('BAAI/bge-small-zh-v1.5'); print('BGE_PREFETCH_OK')"

# 核查 Drive 上冻结的 v0.6-R3 BF16 教师权重与 12k BPE 词表
V07_DIR = Path('/content/drive/MyDrive/MeguvizCAC/v070_omniveritas')
for req in ('v0.6.0-r3-polished-teacher_bf16.pt', 'v0.7.0-r2-omniveritas-b4_bili_picky_bpe_12k.json'):
    p = V07_DIR / req
    print(f'Drive asset {req}: exists={p.is_file()} ({p.stat().st_size/(1024**2):.2f} MB)' if p.is_file() else f'WARNING missing {p}')
print('V10_STEP1_READY | Drive:', DRIVE_OUT)


In [ ]:
# Cell 2: 自动同步并校验 80k + 12.7k Gemini Pro/Flash 统一数据包
# 若已将本地 v08_gemini_80k_13kpairs_release.zip (11.59 MB) 上传到 /content/ 或 Drive，将自动校验 SHA256 并解压覆盖；否则检查 Drive 现有数据。
import gzip, hashlib, zipfile
from pathlib import Path

GEM_DIR = Path('/content/drive/MyDrive/MeguvizCAC/v080_zenith/data')
GEM_DIR.mkdir(parents=True, exist_ok=True)

zip_candidates = [
    Path('/content/v08_gemini_80k_13kpairs_release.zip'),
    GEM_DIR / 'v08_gemini_80k_13kpairs_release.zip',
]
found_zip = next((p for p in zip_candidates if p.is_file()), None)
if found_zip is not None:
    sha = hashlib.sha256(found_zip.read_bytes()).hexdigest()
    print(f'[FOUND] {found_zip} ({found_zip.stat().st_size/(1024**2):.2f} MB) sha256={sha[:16]}')
    with zipfile.ZipFile(found_zip, 'r') as zf:
        for name in zf.namelist():
            if name.endswith(('v08-gemini-gold-labels.jsonl.gz', 'v08-gemini-counterfactual-pairs.jsonl')):
                dst = GEM_DIR / Path(name).name
                dst.write_bytes(zf.read(name))
                print(f'  -> extracted {dst} ({dst.stat().st_size/(1024**2):.2f} MB)')
else:
    print('[INFO] 未在 /content/ 检测到新上传的 v08_gemini_80k_13kpairs_release.zip，检查 Drive 现有数据：')

gold_p = GEM_DIR / 'v08-gemini-gold-labels.jsonl.gz'
pairs_p = GEM_DIR / 'v08-gemini-counterfactual-pairs.jsonl'
assert gold_p.is_file() and pairs_p.is_file(), f'Missing Gemini pack under {GEM_DIR}!'
with gzip.open(gold_p, 'rt', encoding='utf-8') as f:
    n_gold = sum(1 for ln in f if ln.strip())
with pairs_p.open('r', encoding='utf-8') as f:
    n_pairs = sum(1 for ln in f if ln.strip())
print(f'[VERIFIED] GEMINI_GOLD={n_gold:,} rows | GEMINI_PAIRS={n_pairs:,} pairs (expected 79,817 / 12,704 for 80k pack)')


In [ ]:
# Cell 3: 启动抗断线后台全量训练 v1.0.0-ApexLoop R1（setsid nohup + 自动 LOFO 审计 + SHA256 双写 Drive）
import json, os, subprocess, time
from pathlib import Path

%cd /content/MeguvizCAC
for k in ('STAGE1_STEPS_OVERRIDE', 'REFINE_STEPS_OVERRIDE', 'QAT_STEPS_OVERRIDE', 'BS_OVERRIDE', 'VERSION', 'OUT_DIR', 'DRIVE_DIR'):
    os.environ.pop(k, None)

PID_FILE = Path('/content/v10_r1.pid')
DONE_JSON = Path('/content/v10_r1_done.json')
LOG_TRAIN = Path('/content/log_v10_r1_full.txt')
LOG_DRV = Path('/content/log_v10_r1_driver.txt')

def _is_alive(pid_f: Path) -> int | None:
    if not pid_f.is_file():
        return None
    try:
        pid = int(pid_f.read_text().strip())
        os.kill(pid, 0)
        return pid
    except Exception:
        return None

pid_now = _is_alive(PID_FILE)
if pid_now is None and not DONE_JSON.is_file():
    DONE_JSON.unlink(missing_ok=True)
    cmd = 'chmod +x meguviz_cac/run_v10_gpu.sh && setsid nohup bash meguviz_cac/run_v10_gpu.sh > /content/v10_r1_nohup.log 2>&1 < /dev/null & echo $! > /content/v10_r1.pid'
    subprocess.run(['bash', '-lc', cmd], check=True)
    time.sleep(3)
    pid_now = _is_alive(PID_FILE)
    print(f'[LAUNCHED] v1.0.0-ApexLoop R1 background driver pid={pid_now}')
elif pid_now is not None:
    print(f'[ATTACHED] Existing v1.0.0-ApexLoop R1 pid={pid_now} still running')

while True:
    if DONE_JSON.is_file():
        print('=== V10_R1_DONE ===')
        print(DONE_JSON.read_text(encoding='utf-8'))
        break
    alive = _is_alive(PID_FILE)
    gpu_line = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu', '--format=csv,noheader'],
        text=True,
    ).strip()
    tail = ''
    if LOG_TRAIN.is_file():
        lines = LOG_TRAIN.read_text(encoding='utf-8', errors='replace').splitlines()
        tail = lines[-1] if lines else ''
    print(f'[{time.strftime("%H:%M:%S")}] pid={alive} | GPU: {gpu_line} | {tail[:160]}', flush=True)
    if alive is None:
        print('[EXIT DETECTED] Driver exited without DONE_JSON — dumping last 40 lines:')
        for lp in (LOG_DRV, LOG_TRAIN, Path('/content/v10_r1_nohup.log')):
            if lp.is_file():
                print(f'--- {lp.name} ---')
                print('\n'.join(lp.read_text(encoding='utf-8', errors='replace').splitlines()[-40:]))
        break
    time.sleep(90)


In [ ]:
# Cell 4: 断线重连状态核查 / 手动补下载单元
import json
from pathlib import Path
from google.colab import files

done = Path('/content/v10_r1_done.json')
drive_dir = Path('/content/drive/MyDrive/MeguvizCAC/v100_apexloop')
if done.is_file():
    info = json.loads(done.read_text(encoding='utf-8'))
    print(json.dumps(info, ensure_ascii=False, indent=2))
    if Path(info['full_zip']).is_file():
        files.download(info['full_zip'])
elif (drive_dir / 'meguvizcac-v1.0.0-r1-apexloop-b6-full.zip').is_file():
    print('Found on Drive:', sorted(p.name for p in drive_dir.iterdir()))
    files.download(str(drive_dir / 'meguvizcac-v1.0.0-r1-apexloop-b6-full.zip'))
else:
    !nvidia-smi --query-gpu=name,utilization.gpu,memory.used,memory.total --format=csv,noheader
    !tail -n 25 /content/log_v10_r1_driver.txt 2>/dev/null || true
    !tail -n 25 /content/log_v10_r1_full.txt 2>/dev/null || true
